In [4]:
import json
import requests
import folium
from folium.plugins import MarkerCluster
import pandas as pd
from shapely.geometry import Point, shape

# 1. Load Data
clinics = pd.read_csv("../data/processed/wa_clinics_geocoded.csv")
demographics_url = "https://raw.githubusercontent.com/plotly/datasets/master/minoritymajority.csv"
wa_pop = pd.read_csv(demographics_url)
wa_pop = wa_pop[wa_pop['STNAME'] == 'Washington'].copy()
wa_pop['FIPS'] = wa_pop['FIPS'].astype(str).str.zfill(5)

# 2. Fetch County GeoJSON
counties_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
geojson_data = requests.get(counties_url).json()
wa_features = [f for f in geojson_data['features'] if f['id'].startswith('53')]
wa_geojson = {"type": "FeatureCollection", "features": wa_features}

# 3. Spatial Join
county_polygons = {f['id']: shape(f['geometry']) for f in wa_features}

def get_county_fips(lat, lon):
    point = Point(lon, lat)
    for fips, poly in county_polygons.items():
        if poly.contains(point):
            return fips
    return None

clinics['FIPS'] = clinics.apply(lambda r: get_county_fips(r['latitude'], r['longitude']), axis=1)

# 4. Metrics & Per-Capita Access
facility_counts = clinics.groupby('FIPS').agg(
    total_facilities=('name', 'count'),
    hospitals=('type', lambda s: s.str.contains('hospital', case=False, na=False).sum())
).reset_index()

metrics_df = pd.merge(wa_pop[['FIPS', 'CTYNAME', 'TOT_POP']], facility_counts, on='FIPS', how='left').fillna(0)
metrics_df['facilities_per_10k'] = ((metrics_df['total_facilities'] / metrics_df['TOT_POP']) * 10000).round(2)
metrics_df['hospitals_per_10k'] = ((metrics_df['hospitals'] / metrics_df['TOT_POP']) * 10000).round(2)

# Merge calculated metrics directly into GeoJSON feature properties for hover tooltips
metrics_lookup = metrics_df.set_index('FIPS').to_dict('index')
for feature in wa_features:
    fips = feature['id']
    props = metrics_lookup.get(fips, {'CTYNAME': 'Unknown', 'TOT_POP': 0, 'total_facilities': 0, 'facilities_per_10k': 0.0})
    feature['properties']['County'] = props['CTYNAME']
    feature['properties']['Population'] = f"{int(props['TOT_POP']):,}"
    feature['properties']['Facilities'] = int(props['total_facilities'])
    feature['properties']['Per10k'] = float(props['facilities_per_10k'])

# 5. Build Polished Folium Map
wa_center = [47.3826, -120.0628]
m = folium.Map(
    location=wa_center, 
    zoom_start=7, 
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", 
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ",
    control_scale=True
)

# Choropleth Layer
choropleth = folium.Choropleth(
    geo_data=wa_geojson,
    name="Facilities per 10k (Deficit Heatmap)",
    data=metrics_df,
    columns=["FIPS", "facilities_per_10k"],
    key_on="feature.id",
    fill_color="YlOrRd_r",
    fill_opacity=0.65,
    line_opacity=0.4,
    line_color="#444444",
    line_weight=1.2,
    legend_name="Healthcare Facilities per 10,000 Residents (Red = High Deficit)"
).add_to(m)

# Interactive Hover Tooltips
folium.GeoJson(
    wa_geojson,
    name="County Data Tooltips",
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['County', 'Population', 'Facilities', 'Per10k'],
        aliases=['County:', 'Total Pop:', 'Facilities Count:', 'Per 10k Residents:'],
        localize=True,
        sticky=False,
        style="""
            background-color: #1e293b;
            color: #f8fafc;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            font-size: 12px;
            padding: 10px;
            border-radius: 6px;
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2);
        """
    )
).add_to(m)

# 30-Mile Buffer (Emergency Care Access)
hospital_layer = folium.FeatureGroup(name="Hospital 30-Mile Emergency Buffer").add_to(m)
for _, row in clinics[clinics['type'].str.contains('hospital', case=False, na=False)].dropna(subset=['latitude', 'longitude']).iterrows():
    folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=48280,
        color='#b91c1c',
        fill=True,
        fill_opacity=0.1,
        weight=1.5,
        dash_array='5, 5',
        tooltip=f"Emergency Access Zone (30 mi): {row['name']}"
    ).add_to(hospital_layer)

# Clustered Facility Markers
marker_cluster = MarkerCluster(name="Individual Facilities").add_to(m)
for _, row in clinics.dropna(subset=['latitude', 'longitude']).iterrows():
    is_hospital = 'hospital' in str(row.get('type', '')).lower()
    color = "red" if is_hospital else "cadetblue"
    icon_type = "hospital-o" if is_hospital else "medkit"
    
    popup_html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; line-height: 1.4;">
        <h4 style="margin: 0 0 4px 0; color: {'#b91c1c' if is_hospital else '#0284c7'};">{row['name']}</h4>
        <span style="display: inline-block; padding: 2px 6px; background-color: #e2e8f0; border-radius: 4px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
            {row.get('type', 'Clinic')}
        </span>
        <p style="margin: 6px 0 0 0; color: #475569;"><b>City:</b> {row.get('city', 'Unknown')}</p>
    </div>
    """
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=280),
        icon=folium.Icon(color=color, icon=icon_type, prefix="fa")
    ).add_to(marker_cluster)

# Floating Summary Badge Card
title_html = """
<div style="
    position: fixed;
    top: 15px;
    left: 55px;
    z-index: 1000;
    background-color: rgba(255, 255, 255, 0.95);
    padding: 14px 18px;
    border-radius: 8px;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
    max-width: 320px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 5px solid #b91c1c;
">
    <h3 style="margin: 0 0 4px 0; color: #0f172a; font-size: 15px; font-weight: 700;">WA Rural Healthcare Access</h3>
    <p style="margin: 0 0 6px 0; color: #475569; font-size: 11.5px; line-height: 1.4;">
        Spatial modeling of primary care facilities and critical hospital coverage radii against county population density.
    </p>
    <div style="font-size: 11px; color: #64748b;">
        <span style="color: #b91c1c; font-weight: bold;">● Red Counties:</span> Severe facility deficit<br>
        <span style="color: #b91c1c; font-weight: bold;">--- Dashed Rings:</span> 30-mile hospital transit zones
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

folium.LayerControl(collapsed=False).add_to(m)

# Export polished map
m.save("../reports/wa_healthcare_deserts_analysis.html")
m


In [5]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
